<center><h1>Sentiment Classification — Representation Baseline</h1></center>

Before jumping into sequence models like RNN/LSTM (which need padded token sequences fed through a
trainable `Embedding` layer, as in the original RNN notebook), it's good practice to establish a
**baseline** using simple, fixed-length text representations. In this notebook we build that baseline
for our sentiment classification assignment with `tf.keras`: we compare **spaCy word embeddings** against
**TF-IDF**, using the exact same neural network classifier and preprocessing for both.

## Load the data to get started

In [2]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 40.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
import numpy as np
import re
import spacy
import tensorflow as tf
from google.colab import files
uploaded = files.upload()
nlp = spacy.load('en_core_web_md')
tf.random.set_seed(42)

data = pd.read_csv('all_data.csv')
data = data.dropna(subset=['review', 'sentiment']).reset_index(drop=True)
data.head()

Saving all_data.csv to all_data (1).csv


,review,sentiment
0,Aditya Ingole Deaf,2
1,I love the app.! There is no issue but if u co...,1
2,"So hard to use. The web app failed, and the mo...",0
3,I hate that the app makes a sound every time s...,1
4,Useless at BSE star MF meet.voice too mych slo...,0


In [4]:
data['sentiment'].value_counts()

,count
sentiment,
2,15302
0,13466
1,11748


### Preprocess the text

In [5]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['clean_text'] = data['review'].map(preprocess_text)
data = data[data['clean_text'].str.len() > 0].reset_index(drop=True)
data.shape

(39804, 3)

### Let's split the data into train/test

In [6]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    data['clean_text'], data['sentiment'], test_size=0.2, random_state=42, stratify=data['sentiment']
)
y_train_arr = np.array(y_train)
y_test_arr = np.array(y_test)
len(x_train), len(x_test)

(31843, 7961)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
x_train_tfidf = tfidf.fit_transform(x_train).toarray().astype('float32')
x_test_tfidf = tfidf.transform(x_test).toarray().astype('float32')
x_train_tfidf.shape

(31843, 5000)

In [8]:
x_train_emb = np.array([doc.vector for doc in nlp.pipe(x_train, batch_size=256)], dtype='float32')
x_test_emb = np.array([doc.vector for doc in nlp.pipe(x_test, batch_size=256)], dtype='float32')
x_train_emb.shape

(31843, 300)

### Now let's build the baseline model with tf.keras

In [9]:
def build_model(input_dim, num_classes=3):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])
    return model

In [10]:
model_tfidf = build_model(input_dim=x_train_tfidf.shape[1])
history_tfidf = model_tfidf.fit(x_train_tfidf, y_train_arr, epochs=8, batch_size=128,
                                 validation_split=0.1, verbose=0)
print("Final training accuracy:", history_tfidf.history['accuracy'][-1])

Final training accuracy: 0.8680996298789978


In [11]:
model_emb = build_model(input_dim=x_train_emb.shape[1])
history_emb = model_emb.fit(x_train_emb, y_train_arr, epochs=8, batch_size=128,
                             validation_split=0.1, verbose=0)
print("Final training accuracy:", history_emb.history['accuracy'][-1])

Final training accuracy: 0.5705562233924866


### Evaluate both baselines

In [12]:
from sklearn.metrics import f1_score, classification_report

test_loss_tfidf, test_acc_tfidf = model_tfidf.evaluate(x_test_tfidf, y_test_arr, verbose=0)
pred_tfidf = np.argmax(model_tfidf.predict(x_test_tfidf, verbose=0), axis=1)
test_f1_tfidf = f1_score(y_test_arr, pred_tfidf, average='macro')
print(f"TF-IDF baseline -> accuracy: {test_acc_tfidf:.4f}, macro F1: {test_f1_tfidf:.4f}")
print(classification_report(y_test_arr, pred_tfidf))

TF-IDF baseline -> accuracy: 0.6904, macro F1: 0.6764
              precision    recall  f1-score   support

           0       0.71      0.74      0.72      2657
           1       0.62      0.50      0.55      2302
           2       0.71      0.79      0.75      3002

    accuracy                           0.69      7961
   macro avg       0.68      0.68      0.68      7961
weighted avg       0.69      0.69      0.69      7961



In [13]:
test_loss_emb, test_acc_emb = model_emb.evaluate(x_test_emb, y_test_arr, verbose=0)
pred_emb = np.argmax(model_emb.predict(x_test_emb, verbose=0), axis=1)
test_f1_emb = f1_score(y_test_arr, pred_emb, average='macro')
print(f"spaCy embeddings baseline -> accuracy: {test_acc_emb:.4f}, macro F1: {test_f1_emb:.4f}")
print(classification_report(y_test_arr, pred_emb))

spaCy embeddings baseline -> accuracy: 0.5663, macro F1: 0.5356
              precision    recall  f1-score   support

           0       0.55      0.67      0.61      2657
           1       0.46      0.27      0.34      2302
           2       0.62      0.70      0.66      3002

    accuracy                           0.57      7961
   macro avg       0.55      0.55      0.54      7961
weighted avg       0.55      0.57      0.55      7961



### How can we predict a new input using these baselines?

In [14]:
def predict(text, model, vectorizer_fn):
    clean = preprocess_text(text)
    vec = vectorizer_fn([clean])
    pred = np.argmax(model.predict(vec, verbose=0), axis=1)[0]
    return {0: 'negative', 1: 'neutral', 2: 'positive'}[pred]

text = "the app is super smooth now after the last update, love it"
predict(text, model_tfidf, lambda t: tfidf.transform(t).toarray().astype('float32'))

'positive'

In [15]:
predict(text, model_emb, lambda t: np.array([doc.vector for doc in nlp.pipe(t)], dtype='float32'))

'negative'

### Conclusion

In this notebook we established a representation baseline for the sentiment classification task, before
moving on to sequence models like the RNN/LSTM notebook. Using the exact same `tf.keras` classifier and
preprocessing for both representations, the results above show which representation, TF-IDF or spaCy
embeddings, gives a stronger fixed-length baseline on this dataset. Any RNN/LSTM model built afterwards
should be compared against this same baseline to check whether the extra complexity (sequence modeling,
padding, trainable embedding layers) is actually worth it for this specific problem.